# Lung Nematic — análisis de defectos topológicos

Estimación de campos nemáticos y detección de **candidatos a defectos ±1/2** en
histología pulmonar (H&E), con un **modelo nulo por permutación** que dice si el
número de defectos supera lo esperable por azar.

**Instrucciones:** ejecuta las celdas de arriba hacia abajo (▶️). El código está
oculto: solo verás títulos y controles. Puedes ajustar los parámetros y volver a
correr *Run analysis* las veces que quieras.

Tiempo aproximado: ~1–2 min por imagen (más si activas el modelo nulo).

In [ ]:
#@title 🔧 Setup — ejecutar una vez { display-mode: "form" }
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/Danpc11/lung-nematic.git"],
    check=True,
)
import lung_nematic
print("Listo. lung_nematic", lung_nematic.__version__)

In [ ]:
#@title 📁 Cargar imágenes — subir o desde Drive { display-mode: "form" }
image_source = "Subir desde la computadora" #@param ["Subir desde la computadora", "Carpeta de Google Drive"]
#@markdown Si eliges Drive, indica la ruta de la carpeta (p. ej. `MyDrive/histologia`).
drive_folder = "MyDrive/lung_histology" #@param {type:"string"}

import os, shutil, glob

IMAGES_DIR = "/content/images"
EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

if os.path.isdir(IMAGES_DIR):
    shutil.rmtree(IMAGES_DIR)
os.makedirs(IMAGES_DIR, exist_ok=True)

if image_source == "Subir desde la computadora":
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, os.path.join(IMAGES_DIR, os.path.basename(name)))
else:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    src = drive_folder
    if not os.path.isabs(src):
        src = os.path.join("/content/drive", src)
    if not os.path.isdir(src):
        raise FileNotFoundError(f"No se encontró la carpeta de Drive: {src}")
    found = [
        p for p in sorted(glob.glob(os.path.join(src, "*")))
        if p.lower().endswith(EXTS)
    ]
    if not found:
        raise FileNotFoundError(f"No hay imágenes {EXTS} en {src}")
    for p in found:
        shutil.copy(p, os.path.join(IMAGES_DIR, os.path.basename(p)))

names = sorted(os.listdir(IMAGES_DIR))
print(f"{len(names)} imagen(es) lista(s):")
for n in names:
    print("  ", n)

In [ ]:
#@title ⚙️ Parámetros del análisis { display-mode: "form", run: "auto" }
#@markdown Escalas de suavizado del campo director (px, separadas por coma).
sigmas_px = "40, 55, 70, 85" #@param {type:"string"}
#@markdown Espaciado de la rejilla de búsqueda de defectos (px). Menor = más fino y lento.
defect_grid_step_px = 24 #@param {type:"slider", min:8, max:64, step:4}
#@markdown Ignora defectos a menos de esta distancia del borde del tejido (px).
min_edge_distance_px = 30 #@param {type:"slider", min:0, max:120, step:5}
#@markdown Percentil mínimo de densidad orientacional para considerar una región.
density_quantile = 0.45 #@param {type:"slider", min:0, max:0.9, step:0.05}
#@markdown Un defecto debe persistir en al menos tantas escalas.
min_scales_for_persistence = 2 #@param {type:"slider", min:1, max:4, step:1}
#@markdown Radio para fusionar detecciones entre escalas (px).
defect_cluster_radius_px = 70 #@param {type:"slider", min:20, max:150, step:5}
#@markdown Escala de la imagen (µm por pixel). Deja 0 si no la conoces.
microns_per_pixel = 0.0 #@param {type:"number"}
save_diagnostic_panel = True #@param {type:"boolean"}

from lung_nematic.config import AnalysisConfig
try:
    _sigmas = tuple(float(s) for s in sigmas_px.split(",") if s.strip())
    config = AnalysisConfig(
        sigmas_px=_sigmas,
        defect_grid_step_px=int(defect_grid_step_px),
        min_edge_distance_px=int(min_edge_distance_px),
        density_quantile=float(density_quantile),
        min_scales_for_persistence=min(int(min_scales_for_persistence), len(_sigmas)),
        defect_cluster_radius_px=float(defect_cluster_radius_px),
        default_microns_per_pixel=(microns_per_pixel or None),
        save_diagnostic_panel=bool(save_diagnostic_panel),
    )
    config.validate()
    print("Parámetros del análisis listos.")
except Exception as error:
    print("Ajusta los parámetros:", error)

In [ ]:
#@title 🎲 Parámetros del modelo nulo { display-mode: "form", run: "auto" }
#@markdown Compara el número observado de defectos contra orientaciones barajadas.
run_null = True #@param {type:"boolean"}
#@markdown Número de permutaciones. Más = p-valor más fino (y más lento).
n_permutations = 199 #@param {type:"slider", min:19, max:999, step:10}
#@markdown Submuestreo para acelerar. 2 reproduce el conteo exacto; más = más rápido y aproximado.
downsample = 2 #@param {type:"slider", min:1, max:6, step:1}
#@markdown "shuffle" baraja los ángulos observados; "uniform" los reemplaza por aleatorios.
null_mode = "shuffle" #@param ["shuffle", "uniform"]
null_seed = 0 #@param {type:"integer"}
print("Modelo nulo:", "ACTIVADO" if run_null else "DESACTIVADO")

In [ ]:
#@title ▶️ Ejecutar análisis { display-mode: "form" }
import os, glob, warnings, shutil
import pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from lung_nematic.io_utils import read_rgb
from lung_nematic.pipeline import analyze_image
from lung_nematic.preprocessing import make_tissue_mask
from lung_nematic.segmentation import segment_nuclei, select_oriented_nuclei
from lung_nematic.null_model import run_null_model, save_null_histogram

RESULTS_DIR = "/content/results"
if os.path.isdir(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

images = sorted(glob.glob("/content/images/*"))
if not images:
    raise RuntimeError("No hay imágenes cargadas. Ejecuta la celda de carga primero.")

keep = ["image_id", "n_nuclei", "n_oriented_nuclei", "global_nematic_order_S",
        "local_S_median", "n_defects_total", "n_plus_half", "n_minus_half",
        "net_topological_charge", "defect_density_mm2", "mean_defect_confidence"]
null_keep = ["null_mean", "null_std", "null_q2_5", "null_q97_5", "z_score",
             "log2_enrichment", "p_two_sided", "direction"]

rows = []
for path in tqdm(images, desc="Analizando"):
    image_id = os.path.splitext(os.path.basename(path))[0]
    meta = {"filename": os.path.basename(path), "image_id": image_id,
            "group": "lung", "microns_per_pixel": config.default_microns_per_pixel}
    summary = analyze_image(path, meta, RESULTS_DIR, config)
    row = {k: summary.get(k) for k in keep}

    if run_null:
        rgb = read_rgb(path)
        mask, hed = make_tissue_mask(rgb)
        _, nuclei = segment_nuclei(mask, hed, config)
        oriented = select_oriented_nuclei(nuclei, config)
        nres = run_null_model(
            oriented, mask, config,
            n_permutations=int(n_permutations),
            downsample=int(downsample),
            mode=null_mode, seed=int(null_seed),
        )
        save_null_histogram(
            nres, os.path.join(RESULTS_DIR, image_id, f"{image_id}_null_hist.png"),
            title=image_id,
        )
        for k in null_keep:
            row[k] = nres[k]
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, "summary_with_null.csv"), index=False)
print("Listo.")
summary_df

In [ ]:
#@title 📊 Ver resultados { display-mode: "form" }
import glob, os
import pandas as pd
from IPython.display import Image as IPyImage, display

overlays = sorted(glob.glob("/content/results/*/*_nematic_overlay.png"))
for overlay in overlays:
    image_id = os.path.basename(os.path.dirname(overlay))
    print("=" * 60); print(image_id)
    display(IPyImage(overlay, width=900))
    hist = os.path.join(os.path.dirname(overlay), f"{image_id}_null_hist.png")
    if os.path.exists(hist):
        display(IPyImage(hist, width=560))

defect_files = sorted(glob.glob("/content/results/*/*_defects.csv"))
if defect_files:
    all_defects = pd.concat(
        [pd.read_csv(f) for f in defect_files], ignore_index=True
    )
    print(f"Candidatos a defecto (todas las imágenes): {len(all_defects)}")
    display(all_defects)

summary_path = "/content/results/summary_with_null.csv"
if os.path.exists(summary_path):
    print("Resumen:")
    display(pd.read_csv(summary_path))

In [ ]:
#@title ⬇️ Descargar resultados (zip) { display-mode: "form" }
import shutil
from google.colab import files
archive = shutil.make_archive("/content/lung_nematic_results", "zip", "/content/results")
files.download(archive)